# Análise de Modelos de Regressão Linear (Portfólio)

Este notebook carrega o dataset `Precos_de_casas.csv` (ou `Preços_de_casas.csv`) e realiza: EDA, limpeza, split treino/teste, treinamento de um modelo linear, avaliação (RMSE, MAE, R²), summary OLS para interpretação estatística e visualizações.

Objetivo: produzir um notebook limpo e reproduzível adequado para inclusão em um portfólio profissional.

In [ ]:
# Imports e configurações
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.api as sm

sns.set_style('whitegrid')
%matplotlib inline

## 1) Detectar/renomear arquivo de dados (se necessário)
O repositório pode conter o arquivo com acento (`Preços_de_casas.csv`). Para evitar problemas em alguns sistemas, criamos uma cópia sem acento `Precos_de_casas.csv` caso ainda não exista.

In [ ]:
candidates = ['Precos_de_casas.csv','Preços_de_casas.csv','Precos_de_casas .csv','precos_de_casas.csv']
repo_root = Path('.')
csv_path = None
for name in candidates:
    p = repo_root / name
    if p.exists():
        csv_path = p
        break

# Se a versão sem acento não existe, e existe a com acento, criamos a cópia sem acento
if csv_path is None and (repo_root / 'Preços_de_casas.csv').exists():
    src = repo_root / 'Preços_de_casas.csv'
    dst = repo_root / 'Precos_de_casas.csv'
    try:
        import shutil
        shutil.copy(src, dst)
        csv_path = dst
        print(f'Criada cópia sem acento em: {dst}')
    except Exception as e:
        print('Falha ao copiar arquivo:', e)

# fallback: procurar qualquer CSV no top-level
if csv_path is None:
    csvs = list(repo_root.glob('*.csv'))
    csv_path = csvs[0] if csvs else None

if csv_path is None:
    raise FileNotFoundError('Nenhum arquivo CSV encontrado no repositório. Adicione Precos_de_casas.csv ou Preços_de_casas.csv.')

print('Usando:', csv_path)
# tentar leitura com encodings comuns
try:
    df = pd.read_csv(csv_path)
except Exception:
    try:
        df = pd.read_csv(csv_path, encoding='latin1')
    except Exception as e:
        raise e

df.shape

## 2) EDA inicial e verificação dos dados

In [ ]:
# Visão geral
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print('Missing por coluna:
', df.isna().sum())
print('Linhas duplicadas:', df.duplicated().sum())

## 3) Limpeza e pré-processamento rápido
- Renomear colunas para conveniência (remover espaços/acentos)
- Remover linhas sem target
- Dummify variáveis categóricas

In [ ]:
# Normalizar nomes de colunas
def normalize_col(c):
    return c.strip().replace(' ', '_').replace('ç','c').replace('ã','a').replace('é','e').replace('ó','o').replace('í','i')

df = df.rename(columns=lambda c: normalize_col(str(c)))
# Tentar identificar coluna alvo (preco)
target_candidates = [c for c in df.columns if 'preco' in c.lower() or 'preco_de_venda' in c.lower() or 'preco'==c.lower()]
if len(target_candidates)==0:
    raise ValueError('Nao encontrei coluna alvo (preco). Colunas: ' + ','.join(df.columns))
ycol = target_candidates[0]
print('Coluna alvo:', ycol)

# Remover linhas sem target
df = df.dropna(subset=[ycol])

# Remover coluna Id se existir
for candidate in ['Id','id','ID']:
    if candidate in df.columns:
        df = df.drop(columns=[candidate])

# Dummify objetos
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df.shape

## 4) Split treino / teste

In [ ]:
X = df.drop(columns=[ycol])
y = df[ycol].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Treino:', X_train.shape, 'Teste:', X_test.shape)

## 5) Treinar LinearRegression (scikit-learn) e avaliar

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f'LinearRegression - RMSE: {rmse:.2f}, MAE: {mae:.2f}, R2: {r2:.4f}')

## 6) Summary OLS (statsmodels) — interpretação estatística
Observação: OLS dá mais informações sobre significância dos coeficientes e intervalos de confiança. Aqui usamos todas as features (pode ser pesado).

In [ ]:
# Usar X com constante
X_const = sm.add_constant(X)
model_ols = sm.OLS(y, X_const).fit()
print(model_ols.summary())

## 7) Visualizações: Predito vs Real e Resíduos

In [ ]:
# Predito x Real
plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred, alpha=0.6)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('Real')
plt.ylabel('Predito')
plt.title('Predito vs Real')
plt.show()

# Resíduos
residuals = y_test - y_pred
plt.figure(figsize=(7,6))
sns.histplot(residuals, kde=True)
plt.title('Distribuição dos Resíduos')
plt.show()

plt.figure(figsize=(7,6))
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predito')
plt.ylabel('Residuo')
plt.title('Resíduos vs Predito')
plt.show()

## 8) Conclusões rápidas e próximos passos
- Documente aqui as principais métricas (RMSE, MAE, R²) e interprete os coeficientes do modelo OLS.
- Considere testar regularização (Ridge, Lasso) e modelos não-lineares (RandomForest) com validação cruzada para comparar performance.
- Inclua uma seção curta de 'Resumo Executivo' com 3-5 linhas para recrutadores.

In [ ]:
# Opcional: salvar previsões em csv
out = X_test.copy()
out['y_true'] = y_test
out['y_pred'] = y_pred
out.to_csv('predicoes_test.csv', index=False)
print('Salvo: predicoes_test.csv')